# CSCI 362 · Module 9: Python and pandas for Data Analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HeyAvijitRoy/csci-362-databses-data-mining/blob/main/Module%209%20-%20Python%20and%20Pandas/CSCI362_Module09_Student.ipynb)

Professor [Avijit Roy](https://avijitroy.com/teaching/)

**Monday, November 9, 2026 · 75 minutes**  
**Continue Wednesday, November 11, 2026**

This notebook is the lecture, demonstration, and practice space. No separate slides are needed. Follow **predict -> run -> inspect -> practice -> explain**. The detailed code explanations are references to revisit after each demonstration; you do not need to read every bullet during class. Check Brightspace for the current meeting schedule.

**Question:** When were observed shared-bike rental counts highest, and how did they vary by working day?

By the end, you can read a CSV, explain a DataFrame's rows and columns, select and filter records, create columns, summarize groups, and use `transform` to attach group statistics to original rows.

Connect to SQL: a DataFrame resembles a query result; selection resembles `SELECT`; filtering resembles `WHERE`; aggregation resembles `GROUP BY`; group transforms resemble window calculations.

### Monday route
| Minutes | Work |
|---|---|
| 0–7 | Colab setup, question, dataset grain |
| 7–17 | Python essentials and tiny DataFrame |
| 17–27 | CSV loading and inspection |
| 27–40 | Selection, filtering, Exercise 1 |
| 40–50 | New columns, summaries, Exercise 2 |
| 50–60 | Groupby and Exercise 3 |
| 60–68 | Transform demonstration |
| 68–73 | Exercise 4 in pairs |
| 73–75 | Exit ticket; save notebook |

If time is tight, begin Exercise 4 Wednesday. The Wednesday section follows the exit ticket.

## 1 · Choose Colab or VS Code

### Google Colab
1. Open https://colab.research.google.com/ and choose **File → Upload notebook**. Select this `.ipynb`.
2. Save a copy in Drive so your edits persist. Connect to a Python runtime; a CPU runtime is enough.
3. Run the loading cell unchanged: it reads `hour.csv` directly from public GitHub when no local copy is present. No CSV upload or Google Drive mount is required.
4. Run cells in order with Shift+Enter. Read text before executing code. Code cells share variables in memory.
5. Before submission, restart the kernel/session and run all cells to check reproducibility. Download your `.ipynb` with outputs. Double-click the **My interpretation** and exit-ticket text cells to enter written responses.

A kernel/session restart clears variables; rerun the cells from the top to reload the dataset. Files created or uploaded in a Colab runtime are temporary. Download exported files before disconnecting and deleting the runtime. Saving a notebook in Drive does not automatically save runtime files; mounting Drive is not required for this lesson.

### VS Code on your computer (Windows or macOS)
Use the [VS Code setup guide](VS_Code_Setup.md) for Python installation, extensions, package setup, and troubleshooting. If you downloaded only this notebook, get the guide and other module files from the [Module 9 GitHub folder](https://github.com/HeyAvijitRoy/csci-362-databses-data-mining/tree/main/Module%209%20-%20Python%20and%20Pandas).

1. Install Python 3.13 and desktop VS Code. In VS Code, install Microsoft's **Python** and **Jupyter** extensions. The Python extension supports the language; Python itself must also be installed.
2. Open the extracted **Module 9 - Python and Pandas** folder with **File > Open Folder**. Open **Terminal > New Terminal** and create the environment and install packages using the commands for your system below.
3. Open this notebook, click **Select Kernel > Python Environments**, and choose the folder's **`.venv`** (use **Select Another Kernel** first if shown).
4. Predict each example's result, then run cells from the top with **Shift+Enter**. Leave the CSV-loading cell unchanged and `UPLOAD_WITH_DIALOG = False`. It uses a local CSV or public GitHub; no Drive mount is needed.
5. Double-click text cells to enter interpretations. Save with **Ctrl+S** (Windows) or **Cmd+S** (macOS). Exported CSV files are saved locally, normally in the Module 9 folder. Before submitting, restart the kernel, run all cells, and save again. Follow Brightspace for submission directions.

**Windows (PowerShell terminal, from the Module 9 folder):**
```powershell
py -3.13 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```

**macOS (terminal, from the Module 9 folder):**
```bash
python3.13 -m venv .venv
./.venv/bin/python -m pip install -r requirements.txt
```

Run these setup commands in the terminal, not in notebook cells. You only need to create the environment and install packages once. On later visits, select the same kernel and run the notebook from the top. See the setup guide if Python is not installed yet or a command is not recognized.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)

**What this code does**
- `import` makes a library available. A library is a collection of code someone has already written. `import pandas as pd` gives pandas the short name `pd`; later, `pd.read_csv(...)` means “use pandas to read a CSV.”
- `from pathlib import Path` brings in the tool used to describe file locations. `display` is a notebook tool that shows tables neatly.
- A dot means “look inside this object or library.” `pd.__version__` retrieves pandas' version. Parentheses `(...)` call a function, such as `print(...)`.
- `sys.version.split()[0]` first splits the Python version text into a list of pieces, then `[0]` takes the first piece. Python starts counting positions at zero. The comma-separated arguments to `print` are displayed together. This cell confirms the environment; you do not need to memorize its version syntax.

## 2 · Python essentials: values, collections, conditions, functions
Python uses `=` to assign and `==` to compare. A string uses quotes. Lists use square brackets; dictionaries map keys to values. Indexing starts at zero. Indentation defines the body of a loop, condition, or function. A variable's name is case-sensitive.

Predict the output before running. You need these building blocks, not a full Python course, to begin pandas.

In [ ]:
rentals = 120                   # integer
average_rentals = 95.5          # floating-point number
location = 'Capital Bikeshare'  # string
busy = rentals > average_rentals  # Boolean
hours = [8, 12, 17]             # list
record = {'hour': 8, 'rentals': 120}  # dictionary
print(location, busy, hours[0], record['rentals'])
print(10 / 4, 10 // 4, 10 % 4)

def difference(observed, baseline):
    return observed - baseline

for hour in hours:
    if hour >= 16:
        print(hour, 'late afternoon/evening')
print(difference(rentals, average_rentals))

**What this code does**
- `rentals = 120` stores a number under a name. The `=` assigns a value; it does not ask whether two things are equal. Text such as `'Capital Bikeshare'` needs quotes. A comment begins with `#` and is not executed.
- `rentals > average_rentals` compares values and produces `True` or `False`. `/` divides, `//` gives floor division, and `%` gives the remainder. The three displayed results are `2.5`, `2`, and `2`.
- `[8, 12, 17]` **creates a list**. In `hours[0]`, the brackets instead **select an item** from that list: position 0 is 8. `hours[2]` would be 17.
- `{'hour': 8, 'rentals': 120}` creates a dictionary with named entries. `record['rentals']` looks up the entry with that text key, giving 120. The quotes identify the key as text rather than another variable name.
- `def difference(observed, baseline):` defines a reusable function. Its indented `return` sends back the subtraction result. Calling it with `rentals` and `average_rentals` returns 24.5.
- `for hour in hours:` visits the three list items. The indented `if` checks each one; only 17 meets `>= 16`. Colons begin these blocks; indentation tells Python which statements belong inside them.

**Check your understanding:** What is the difference between `hours[0]` and `hours[2]`? Why is `'120'` different from `120`? How would you change the condition to include only hour 17?

A DataFrame is a two-dimensional labeled table. A Series is one labeled column. The index labels rows; it is not automatically a database primary key. pandas usually works on whole columns, so you rarely need a Python loop for arithmetic on rows.

In [ ]:
tiny = pd.DataFrame({'hr': [8, 12, 17], 'cnt': [120, 80, 180]})
display(tiny)
print(type(tiny), type(tiny['cnt']))
display(tiny['cnt'] * 2)  # vectorized: multiply every value

**What this code does — brackets in detail**
- `pd.DataFrame(...)` creates a table. Inside its parentheses, the dictionary `{'hr': [8, 12, 17], 'cnt': [120, 80, 180]}` supplies column names and lists of values. The first value in each list forms the first row. Both lists must have the same length.
- `tiny['cnt']` asks the table for **one column**, using its name in quotes. The result is a Series (a single labeled column).
- `tiny[['hr', 'cnt']]` would ask for **multiple columns**. Read this from the inside out: `['hr', 'cnt']` is a list of names; the outer `tiny[ ... ]` passes that list to the table to select those columns. It is not indexing twice into the same column.
- `type(...)` reports what kind of object you have. `display(...)` shows the table or Series. `tiny['cnt'] * 2` doubles every value in that column without writing a loop. It displays a result and does not overwrite `tiny`.

## 3 · Our public dataset and its grain
We use the **UCI Bike Sharing dataset**, hourly Capital Bikeshare rental counts for the Washington, D.C. system in 2011–2012. This is real historical data, not synthetic data or current demand.

**One row = one observed date/hour for the system**, not one rider, trip, station, or bike. The packaged CSV contains 17,379 rows and 17 original columns. Not every possible date/hour has a row; do not silently treat absent hours as zero.

**Attribution:** Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5W894. UCI lists CC BY 4.0: https://creativecommons.org/licenses/by/4.0/. The original `Readme.txt` is included unchanged. Associated paper: Fanaee-T & Gama (2013), https://doi.org/10.1007/s13748-013-0040-3.

| Column | Meaning |
|---|---|
| `instant` | Source record ID; not a predictor of interest |
| `dteday`, `hr` | Date and hour (0–23) |
| `yr`, `mnth` | 0 = 2011, 1 = 2012; month 1–12 |
| `weekday` | 0 = Sunday through 6 = Saturday |
| `workingday` | 1 = neither weekend nor holiday; otherwise 0 |
| `holiday` | Holiday indicator, 0/1 |
| `season` | Current UCI dictionary: 1 winter, 2 spring, 3 summer, 4 fall |
| `weathersit` | 1 clear/clouds; 2 mist/cloudy; 3 light precipitation; 4 severe precipitation |
| `temp`, `atemp` | Normalized temperature and apparent temperature; not raw Celsius |
| `hum`, `windspeed` | Normalized humidity and wind speed; not raw measurements |
| `casual`, `registered` | Rental counts by user category |
| `cnt` | Total rental count: casual + registered |

We leave weather measures normalized. The unchanged source readme and current UCI page differ in season labels and temperature normalization. We use the current UCI season labels and perform no Celsius conversion in this lesson. Category numbers are labels, so their numeric mean is not a meaningful weather or season summary.

### Read a CSV: GitHub or a local file
**Default:** run the next cell unchanged. It uses a supplied local `hour.csv` if one is available; otherwise it reads the CSV directly from the course's public GitHub repository. In a fresh Colab runtime, the GitHub link works without uploading a CSV or mounting Google Drive. Internet access is required for the GitHub route.

**Optional upload:** if GitHub is unavailable, upload `hour.csv` through Colab's folder sidebar and rerun the cell. Alternatively, change `UPLOAD_WITH_DIALOG` to `True` and select `hour.csv` in the dialog. Leave it `False` for local Jupyter.

**Optional path:** set `CSV_PATH` to your file, for example `Path('/content/hour.csv')`. With `CSV_PATH = None`, the cell checks the current folder, `/content`, the parent folder, and the Module 9 folder before using GitHub.

`pd.read_csv` reads either a local file or a URL into a DataFrame. `parse_dates` asks pandas to interpret the date column. Relative paths start from the runtime's current working folder.
**Checkpoint:** confirm 17,379 rows and 17 columns after loading. Point to one row and state its unit before continuing.


In [ ]:
UPLOAD_WITH_DIALOG = False
CSV_PATH = None  # Optional: Path('/content/hour.csv') / your actual file path
CSV_URL = (
    'https://raw.githubusercontent.com/HeyAvijitRoy/'
    'csci-362-databses-data-mining/main/'
    'Module%209%20-%20Python%20and%20Pandas/hour.csv'
)

if UPLOAD_WITH_DIALOG:
    from google.colab import files
    uploaded = files.upload()
    if 'hour.csv' not in uploaded:
        raise ValueError('Select the supplied hour.csv file.')
    CSV_PATH = Path('hour.csv')

if CSV_PATH is None:
    candidates = [Path('hour.csv'), Path('/content/hour.csv'), Path('../hour.csv'), Path('Module 9 - Python and Pandas/hour.csv')]
    CSV_PATH = next((p for p in candidates if p.is_file()), None)
if CSV_PATH is not None and not Path(CSV_PATH).is_file():
    raise FileNotFoundError('Check CSV_PATH or set it to None to load from GitHub.')

CSV_SOURCE = CSV_PATH if CSV_PATH is not None else CSV_URL
bikes = pd.read_csv(CSV_SOURCE, parse_dates=['dteday'])
print('Loaded from:', CSV_SOURCE)
print('Rows, columns:', bikes.shape)
display(bikes.head())


**What this code does — loading step by step**
- `False` turns the optional upload dialog off. `None` means “no file path has been chosen yet.” Beginners can leave both settings unchanged.
- `CSV_URL` stores the public GitHub address of the supplied dataset. The strings inside parentheses join into one URL; `%20` represents a space in the folder name.
- An `if` runs its indented block only when its condition is true. If you turn the dialog on in Colab, `files.upload()` asks you to select files. `uploaded` records their filenames; `'hour.csv' not in uploaded` checks whether the required name is absent.
- `candidates = [...]` creates a list of possible file paths. `Path('hour.csv')` means a file in the current folder; `../hour.csv` means look for the CSV one folder above the current folder.
- `next((p for p in candidates if p.is_file()), None)` selects the first existing file, or `None` if none exists. This setup code lets the same notebook work locally and in Colab.
- `raise FileNotFoundError(...)` stops with a helpful message if a chosen file path does not exist. Set `CSV_PATH` back to `None` to use automatic loading.
- `CSV_SOURCE` chooses the local path when available and the GitHub URL otherwise. No Drive connection is needed. The printed source shows which was used.
- `pd.read_csv(CSV_SOURCE, parse_dates=['dteday'])` reads the CSV. `parse_dates=` names an option; `['dteday']` identifies the date column to parse. The result is assigned to `bikes`.
- `bikes.shape` retrieves the pair (row count, column count), without parentheses. `bikes.head()` calls a method, with parentheses, to preview the first five rows.


### Inspect before analyzing
`head` previews rows; `info` reports types and nonmissing counts; `describe` summarizes numeric columns. It will include ID and category columns, so choose relevant measures when interpreting. `size` counts rows within a group; `count` counts nonmissing values in a chosen column. Missing values require an explicit decision, which we explore in the next module.

In [ ]:
bikes.info()
display(bikes[['cnt', 'casual', 'registered']].describe().round(2))
print('Date range:', bikes['dteday'].min().date(), 'to', bikes['dteday'].max().date())
print('Missing cells:', bikes.isna().sum().sum())
print('Duplicate date/hour pairs:', bikes.duplicated(['dteday', 'hr']).sum())
assert bikes.shape == (17379, 17)
assert bikes['instant'].is_unique
assert (bikes['casual'] + bikes['registered']).equals(bikes['cnt'])

**What this code does**
- `bikes.info()` prints column names, types, and nonmissing counts. It helps confirm that the file loaded as expected.
- In `bikes[['cnt', 'casual', 'registered']]`, the inner brackets build a list of three column names; the outer brackets select those columns as a smaller table. `.describe()` summarizes that table; `.round(2)` rounds its displayed summary to two decimal places.
- `bikes['dteday'].min()` finds the earliest date; `.max()` finds the latest. `.date()` displays the date without its time component.
- `bikes.isna()` creates a table of missing/not-missing flags. The first `.sum()` counts missing cells in each column; the second adds those counts together.
- `.duplicated(['dteday', 'hr'])` checks repeated **combinations** of date and hour. Its `.sum()` counts repeats after the first occurrence.
- `assert` is a check: execution stops if the statement is false. `(17379, 17)` is a tuple (an ordered pair); `==` compares it with the shape. `.is_unique` checks IDs. `.equals(...)` checks that the calculated column matches `cnt` throughout. These checks verify the supplied file, rather than changing it.

## 4 · Selection, filtering, sorting
`bikes['cnt']` returns a Series. `bikes[['hr', 'cnt']]` returns a DataFrame. `loc` selects row labels or Boolean conditions and column labels; `iloc` selects integer positions. `iloc[0:3]` includes positions 0, 1, 2 (stop is excluded).

Use `&` for AND, `|` for OR, and `~` for NOT on pandas Boolean Series. Parenthesize each condition. Python's `and` and `or` do not combine whole Series. A filter chooses rows; it does not aggregate them.

In [ ]:
display(bikes[['dteday', 'hr', 'cnt']].head(3))
display(bikes.iloc[0:3, 0:4])
busy_mask = (bikes['workingday'] == 1) & (bikes['cnt'] >= 500)
busy_hours = bikes.loc[busy_mask, ['dteday', 'hr', 'cnt']].copy()
display(busy_hours.sort_values(['cnt', 'dteday', 'hr'], ascending=[False, True, True]).head(10))
print('Qualifying observed hours:', len(busy_hours))

**What this code does — choosing columns and rows**
- `bikes[['dteday', 'hr', 'cnt']]` selects a list of column names. `.head(3)` displays the first three rows.
- `bikes.iloc[0:3, 0:4]` selects by **position**. The comma separates rows (before it) from columns (after it). `0:3` means positions 0, 1, 2; `0:4` means columns at positions 0, 1, 2, 3. The stopping position is excluded.
- `bikes['workingday'] == 1` produces one True/False answer per row. So does `bikes['cnt'] >= 500`. Parentheses enclose each condition, and `&` combines them so a row must meet both. `busy_mask` stores those answers, not the selected rows.
- `bikes.loc[busy_mask, ['dteday', 'hr', 'cnt']]` means “keep rows whose mask is True, and these three named columns.” The outer brackets belong to `.loc`; the inner brackets are the list of column names. `.copy()` makes a separate table we can safely work with.
- `.sort_values(['cnt', 'dteday', 'hr'], ascending=[False, True, True])` sorts by count descending, then date ascending, then hour ascending to settle ties. The two lists line up in order. `.head(10)` shows just ten results. `len(busy_hours)` counts the qualifying rows (observed hours).

### Exercise 1 · Monday: select and filter
Find records from **2012**, with hours **7, 8, or 9**, on **working days**. Save only `dteday`, `hr`, `cnt` as `morning`. Show five rows and report the number of rows. Hint: `.isin([7, 8, 9])`. Explain why this row count is not a count of trips.

Write your code below, then answer the interpretation questions in **My interpretation**.

In [ ]:
# Your work here


**My interpretation:**

_Write your response here._


**Hints and syntax review: try the exercise first**
`['yr']`, `['hr']`, and `['workingday']` each select one column. `==` tests equality; `.isin([7, 8, 9])` checks membership in a list. Join the three parenthesized conditions with `&`. In `.loc[condition, ['dteday', 'hr', 'cnt']]`, the comma separates the row condition from the list of output columns. `.copy()` gives the selected table its own data. `len(morning)` counts hourly records; it does not add their rental counts. Try explaining each bracket pair aloud.

## 5 · Transform columns and summarize values
Create a column by assigning a Series of matching length. `.assign()` returns a new DataFrame and keeps the original intact. We compute a **percentage of rentals**, not a percentage of unique people: the data do not identify riders.

All observed `cnt` values here are positive. In other data, check denominators before dividing. Do not modify filtered views with chained indexing; use `.loc` or `.copy()` as shown above.

In [ ]:
assert bikes['cnt'].gt(0).all()
analysis = bikes.assign(
    casual_pct=100 * bikes['casual'] / bikes['cnt'],
    year=2011 + bikes['yr'],
    humidity_pct=100 * bikes['hum']
)
display(analysis[['dteday', 'hr', 'cnt', 'casual_pct', 'humidity_pct']].head().round(2))
print('Total rentals:', bikes['cnt'].sum())
print('Mean rentals per observed hour:', round(bikes['cnt'].mean(), 2))
print('Median rentals per observed hour:', bikes['cnt'].median())

**What this code does**
- `bikes['cnt'].gt(0)` checks whether each count is greater than zero. `.all()` asks whether every answer is True. The assertion protects the division that follows.
- `.assign(...)` returns a table containing the original columns plus the named new columns. The comma separates definitions. `casual_pct=...` names the new column; it is a keyword argument to `assign`, not a comparison.
- `100 * bikes['casual'] / bikes['cnt']` divides each row's casual count by that same row's total and multiplies by 100. These operations work across whole columns.
- `2011 + bikes['yr']` converts 0/1 into 2011/2012. `100 * bikes['hum']` expresses fractional humidity as a percentage. `analysis = ...` stores this expanded table; `bikes` stays as loaded.
- The double brackets select columns for a preview. `.sum()` adds counts; `.mean()` computes their arithmetic average; `.median()` finds their middle value. `round(number, 2)` rounds a single number, whereas `.round(2)` is a method on a pandas object. The reported mean is per observed hour, not per rider.

### Exercise 2 · Monday: a percentage needs a denominator
Compute the percentage of **all rentals** that were registered-user rentals. Compare it with the mean of hourly registered-user percentages. Are they equal? Explain why large and small hours should not automatically receive equal weight when answering the all-rentals question.

Write your code below, then answer the interpretation questions in **My interpretation**.

In [ ]:
# Your work here


**My interpretation:**

_Write your response here._


**Hints and syntax review: try the exercise first**
Select the registered and total columns separately with `['registered']` and `['cnt']`. Calling `.sum()` on each column first produces two totals; dividing them gives the overall share. In the alternative expression, the parentheses around the per-row calculation ensure `.mean()` acts on the resulting hourly percentages. No rows are grouped here. `round(value, 2)` and `print(...)` format and show the two answers; they do not alter the source file. Explain the denominator before interpreting a percentage.

## 6 · Groupby: split, apply, combine
Split records into groups, apply an aggregation, combine into a smaller table. `agg` can create several named statistics at once. `as_index=False` leaves grouping labels as ordinary columns.

Question: which hour has the largest **average count per observed date/hour**? Total counts and means answer different questions when group sizes differ. Always report the number of observations alongside averages.

SQL connection: `SELECT hr, COUNT(*), AVG(cnt), SUM(cnt) FROM bikes GROUP BY hr;`

In [ ]:
hourly = bikes.groupby('hr', as_index=False).agg(
    observed_hours=('cnt', 'size'),
    mean_rentals=('cnt', 'mean'),
    median_rentals=('cnt', 'median'),
    total_rentals=('cnt', 'sum')
)
display(hourly.sort_values('mean_rentals', ascending=False).head(5).round(2))
assert hourly['total_rentals'].sum() == bikes['cnt'].sum()

**What this code does — reading groupby syntax**
- `bikes.groupby('hr', as_index=False)` puts together rows with the same hour code. `'hr'` is one column name, so no list is required. `as_index=False` keeps hour as a normal column in the result.
- `.agg(...)` calculates summaries within each group. In `observed_hours=('cnt', 'size')`, the left side gives the output column's name. The tuple on the right gives the source column and operation. `'size'` counts rows; `'mean'`, `'median'`, and `'sum'` calculate the other summaries.
- Parentheses after `agg` contain its arguments; parentheses around `('cnt', 'mean')` create a pair of instructions. Neither is a row-selection bracket.
- `hourly` has one row per hour code, rather than one row per observed date/hour. Sorting its means descending and taking `.head(5)` shows the five hours with the largest means. Rounding changes this displayed result, not the stored raw data.
- The assertion checks that adding the grouped totals recovers the original total: aggregation should not lose rentals.

### Exercise 3 · Monday: compare working-day groups
Group by `workingday`. Compute observed hours, mean rentals, and total rentals. Which group has the higher mean? Explain why the result alone cannot establish that working-day status causes demand. Reminder: 0 includes weekends **and holidays**.

Write your code below, then answer the interpretation questions in **My interpretation**.

In [ ]:
# Your work here


**My interpretation:**

_Write your response here._


**Hints and syntax review: try the exercise first**
`groupby('workingday')` splits rows into two categories. In `.agg(mean_rentals=('cnt', 'mean'))`, the output name is on the left; the source column and operation are the pair on the right. `as_index=False` leaves the category as a column. To identify the higher mean, compare the two summary rows. You can also use `.idxmax()` to find the row label with the largest mean and `.loc` to retrieve its category. A group summary describes an association, not a controlled experiment.

## 7 · Group transform: keep the original rows
`agg('mean')` gives one value per group. `transform('mean')` repeats the relevant group mean at every original row, aligned to its index. This lets us compare each observation with its own group's baseline.

Here, the group is **hour of day** across both years. The deviation is an observed count minus the mean for that hour. Positive means above that historical hour's average. This is descriptive, not a forecasting model or a causal estimate. In future predictive work, a baseline must be learned only from training data.

In [ ]:
comparison = bikes.copy()
comparison['hour_mean'] = comparison.groupby('hr')['cnt'].transform('mean')
comparison['above_hour_mean'] = comparison['cnt'] - comparison['hour_mean']
display(comparison[['dteday', 'hr', 'cnt', 'hour_mean', 'above_hour_mean']].head().round(2))
print('Aggregated rows:', len(hourly), '| Transformed rows:', len(comparison))
assert len(comparison) == len(bikes)
assert comparison.index.equals(bikes.index)
assert comparison.groupby('hr')['above_hour_mean'].mean().abs().max() < 1e-9

**What this code does — group averages beside original records**
- `comparison = bikes.copy()` makes a separate table. `comparison['hour_mean'] = ...` uses brackets on the left of `=` to create a new named column.
- On the right, `.groupby('hr')['cnt']` first forms hour groups, then chooses the count column within them. Here the bracket pairs occur in separate steps; this is different from the list-inside-selection pattern `df[['hr', 'cnt']]`.
- `.transform('mean')` computes each group's mean and returns one value for **each original row**, in matching row order. A row at hour 8 receives the hour-8 mean; a row at hour 17 receives the hour-17 mean.
- `comparison['cnt'] - comparison['hour_mean']` subtracts matching row values. A positive deviation means that observed count exceeds its hour's baseline.
- `len(...)` shows that aggregation has 24 rows while transformation keeps 17,379. `.index.equals(...)` checks row-label alignment. The final check groups deviations, takes means and absolute values (`abs`), and requires the largest to be below `1e-9` (0.000000001). This allows tiny computer-rounding differences around zero.

### Exercise 4 · Monday pair task / Wednesday warm-up
Attach `day_mean`, the mean count within each `workingday` group, to every row of a new DataFrame named `day_comparison`. Add `deviation = cnt - day_mean`. Show three rows. State the number of rows returned by aggregation versus transformation. Do not replace missing hours with zero.

Write your code below, then answer the interpretation questions in **My interpretation**.

In [ ]:
# Your work here


**My interpretation:**

_Write your response here._


**Hints and syntax review: try the exercise first**
Begin with `.copy()` to create `day_comparison`. Brackets on the left, such as `day_comparison['day_mean'] = ...`, name a new column. `groupby('workingday')['cnt'].transform('mean')` computes means for the two categories and repeats the appropriate mean beside every original row. Subtraction computes a deviation for each row. Double brackets select several preview columns. `.nunique()` counts distinct category values, while `len(...)` counts rows. Check row count, index alignment, and approximately zero average deviation within each category.

## Monday exit ticket · 2 minutes
Answer in the text cell below:
1. What does one row represent?
2. What is the difference between filtering and grouping?
3. When would you use `transform` instead of `agg`?
4. Name one limitation on the claim “working days cause more rentals.”

Save your notebook for Wednesday; the loading cell can read `hour.csv` from GitHub again. Finish unanswered Monday practice before the next meeting if instructed. Follow Brightspace for submission requirements and deadlines; this package does not create a new graded assignment.

**My exit-ticket answers:**

1. 
2. 
3. 
4.

# Wednesday · November 11 continuation
Suggested 75-minute plan: 0–10 restart/load/retrieval check; 10–25 finish and discuss Monday exercises; 25–40 two-key grouping; 40–55 Exercise 5; 55–65 Exercise 6; 65–72 share claims and limitations; 72–75 preview cleaning and visualization.

Restart the kernel/session and run the notebook from the top before continuing; the loading cell reads from GitHub again if no local CSV is present. You may leave Monday exercise cells unanswered without breaking demonstrations; each demonstration uses only worked-example variables.

## 8 · Group by two keys
Compare working and nonworking days **within the same hour**. This addresses one composition difference, though other differences remain. Sort group labels before interpreting the result. Later we can plot these summaries; today inspect a small table.

In [ ]:
profile = bikes.groupby(['workingday', 'hr'], as_index=False).agg(
    observed_hours=('cnt', 'size'), mean_rentals=('cnt', 'mean')
)
display(profile.loc[profile['hr'].isin([8, 12, 17])].round(2))
assert len(profile) == 48

**What this code does**
- `groupby(['workingday', 'hr'])` uses a **list of two grouping columns** inside the function's parentheses. Each distinct pair forms a group: for example, workingday 1 at hour 8 is separate from workingday 0 at hour 8.
- Named aggregation counts the rows and averages `cnt` for each pair. With both workingday categories and all 24 hours present, `profile` contains 48 rows.
- `profile['hr'].isin([8, 12, 17])` checks whether each row's hour is in that list. It returns a Boolean mask, which `profile.loc[...]` uses to select rows. Because no comma or column selector is supplied to `loc`, all columns are kept.
- `.round(2)` makes the displayed summary easier to read. Compare workingday groups within the same hour, and inspect their observation counts as well as their averages. The assertion confirms the expected number of groups.

### Exercise 5 · Wednesday: strongest working-day hours
For `workingday == 1`, group by `hr`, computing observed hours and mean rentals. Find the top three hours by mean rentals; break any ties by hour ascending. Report the group sizes and write a two-sentence conclusion including the historical scope and a limitation.

Write your code below, then give the two-sentence conclusion in **My interpretation**.

In [ ]:
# Your work here


**My interpretation:**

_Write your response here._


**Hints and syntax review: try the exercise first**
`bikes.loc[bikes['workingday'] == 1]` selects only working-day rows, keeping every column. Group that selected table by hour and use named aggregations to count observations and average rentals. `sort_values(['mean_rentals', 'hr'], ascending=[False, True])` matches two sort columns to two directions: descending mean, then ascending hour to break ties. `.head(3)` returns only the first three sorted rows; it does not mean hours 0, 1, and 2. Include the observed-hour counts in your interpretation.

### Exercise 6 · Wednesday: contextual baseline and export
Use `transform` with both `workingday` and `hr` to attach a baseline called `context_mean` to each original row. Compute `context_deviation`. Show the five largest positive deviations (sort deviation descending, then date and hour ascending to break ties), with `dteday`, `hr`, `cnt`, `context_mean`, and `context_deviation`. Export those five rows to `top_deviations.csv` with `index=False`. Explain why this is not proof of an anomaly or a forecast.

Write your code below, then answer the interpretation questions in **My interpretation**.

In [ ]:
# Your work here


**My interpretation:**

_Write your response here._


**Hints and syntax review: try the exercise first**
`groupby(['workingday', 'hr'])` forms groups using a list of two keys. Selecting `['cnt']` afterward chooses the measure, and `transform('mean')` returns one aligned baseline for every original row. Create deviation with column subtraction. The sort uses three columns and matching directions; the following double brackets select five output columns, and `.head(5)` selects five rows. `.to_csv('top_deviations.csv', index=False)` writes a file; the option prevents pandas row labels becoming an extra CSV column. Read the CSV back with `pd.read_csv` and check its five rows and five requested columns. The file is written to the runtime's current folder; download it to keep it.

## 9 · Troubleshooting and quick reference
| Symptom / task | Response |
|---|---|
| `FileNotFoundError` | Check your explicit path, or set `CSV_PATH = None` for automatic loading |
| GitHub/network loading error | Check the connection and rerun; alternatively upload `hour.csv` through the folder sidebar and rerun |
| `NameError` | Run the earlier cell defining that variable; restart and run in order |
| `KeyError` | Check spelling/case with `bikes.columns.tolist()` |
| “truth value of a Series is ambiguous” | Use parenthesized conditions with `&` or `|` |
| Unexpected rows in a slice | `loc` uses labels; `iloc` uses positions |
| New column on a subset | Make `.copy()` or use `.loc` assignment |
| One column / multiple columns | `df['cnt']` / `df[['hr', 'cnt']]` |
| Filter | `df.loc[(df['hr'] >= 7) & (df['hr'] <= 9)]` |
| Summary / row-aligned baseline | `groupby(...).agg(...)` / `groupby(...).transform(...)` |
| Export | `table.to_csv('result.csv', index=False)` |

In Colab, download exported CSV files using the folder sidebar's file menu. Download the completed notebook using **File → Download → Download .ipynb**. Runtime outputs are not a substitute for saving your code.

## Reading before / after class
The included `Reading.md` is the primary short reading. Read sections 1–4 before class if possible; review sections 5–7 before Wednesday. Official references are linked there for optional deeper study.

## Next module bridge
We checked missing cells and duplicate date/hour pairs, but did not invent cleaning problems in this file. Next: discuss absent timestamps versus missing cells, inspect data types and unusual values, then visualize distributions. Avoid interpreting normalized measures as raw units or deriving causal claims from group differences.